### Install And Import

In [ ]:
!pip install scikit-learn
!pip install tf_keras
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install torch
!pip install sentencepiece
# !pip install tensorflow[and-cuda]




# Install required packages
!pip install -q transformers datasets
!pip install transformers
# Reinstall transformers and tensorflow to ensure proper setup for TF models
# !pip uninstall -y transformers
# !pip install -q transformers==4.41.0



# !pip uninstall -y tf_keras keras keras-nightly keras-preprocessing
# !pip install --upgrade "tensorflow>=2.12" transformers


In [ ]:


import os



# os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Now import your packages
import tensorflow as tf
import tf_keras

In [ ]:


# Check versions
# import tensorflow as tf
from transformers import __version__ as transformers_version

print(f"TensorFlow Version: {tf.__version__}")
print(f"Transformers Version: {transformers_version}")

In [ ]:
### Download Data From Kaggle

#Connect Google drive to colab
# from google.colab import drive
# drive.mount('/gdrive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import transformers


### Data Processing

Load data

In [ ]:
# df = pd.read_csv('labeledTrainData.tsv.zip', sep='\t')

df = pd.read_csv('train_(1)_(2)_(1).csv')

print(df.shape)

In [ ]:
df.sample(n=5)

In [ ]:
#Sentences and labels
sentences = df.Review.values
labels = df.Rating.values

## Tokenize data using Bert Tokenizer

In [ ]:
# from transformers import AutoTokenizer

# Load tokenizer (AutoTokenizer is the modern way)
tokenizer = transformers.AutoTokenizer.from_pretrained('bert-base-uncased')



In [ ]:
#tokenizer.vocab.items()

In [ ]:
tokenized_texts = [tokenizer.tokenize(sent) for sent in sentences]

In [ ]:
sentences[0]

In [ ]:
type(sentences[0])

In [ ]:
len('good')

In [ ]:
len(sentences[0].split(' '))

In [ ]:
#Check tokenized text
print(tokenized_texts[0])

In [ ]:
len(tokenized_texts[0])

In [ ]:
#We will use only first 200 tokens to do classification (this value can be changed)
max_length = 200
tokenized_texts = [sent[:max_length] for sent in tokenized_texts]

In [ ]:
for i in range(len(tokenized_texts)):
    sent = tokenized_texts[i]
    sent = ['[CLS]'] + sent + ['[SEP]']
    tokenized_texts[i] = sent

In [ ]:
print(tokenized_texts[0])

In [ ]:
#Convert tokens into IDs
input_ids = [tokenizer.convert_tokens_to_ids(sent) for sent in tokenized_texts]

In [ ]:
print(input_ids[0])

In [ ]:
#Pad our tokens which might be less than max_length size
input_ids = tf.keras.preprocessing.sequence.pad_sequences(input_ids,
                                                          maxlen=max_length+2,
                                                          truncating='post',
                                                          padding='post')

Split data between training and test

In [ ]:
#80% data will be used for training while 20% will be used for test
trainX, testX, trainY, testY = train_test_split(input_ids, labels,
                                                test_size=0.2, random_state=12345)

Create Attention masks : Attention masks are useful to ignore padding tokens. Mask value will be set to 0 for padding tokens and 1 for actual tokens. We will create mask both for training and test data

In [ ]:
# Create attention masks for training
train_attn_masks = []

# Create a mask of 1s for each token followed by 0s for padding
for seq in trainX:
  seq_mask = [float(i>0) for i in seq]
  train_attn_masks.append(seq_mask)

In [ ]:
# Create attention masks for Test
test_attn_masks = []

# Create a mask of 1s for each token followed by 0s for padding
for seq in testX:
  seq_mask = [float(i>0) for i in seq]
  test_attn_masks.append(seq_mask)

In [ ]:
print(train_attn_masks[100])

### Build Model

In [ ]:
# from transformers import TFBertForSequenceClassification

model = transformers.TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    from_pt=True   # explicitly convert from PyTorch
)

In [ ]:
model.summary()

In [ ]:
import tf_keras
# Use tf_keras optimizer to ensure compatibility with TFBert models in newer TF versions
optimizer = tf_keras.optimizers.Adam(learning_rate=3e-5, epsilon=1e-08, clipnorm=1.0)

In [ ]:
# Define loss and metrics using tf_keras
loss = tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf_keras.metrics.SparseCategoricalAccuracy('accuracy')]

In [ ]:
# !pip install "tensorflow<2.16"

In [ ]:
# 2. Compile the model using tf_keras compatible objects
# This avoids the '_distribute_strategy' error in TF 2.16+
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

### Train Model

In [ ]:
train_x_data = {'input_ids': np.array(trainX), 'attention_mask': np.array(train_attn_masks)}
test_x_data = {'input_ids': np.array(testX), 'attention_mask': np.array(test_attn_masks)}

In [ ]:
# Training the model with the prepared data
model.fit(
    train_x_data,
    trainY,
    validation_data=(test_x_data, testY),
    batch_size=16,
    epochs=2
)

In [ ]:
testData = pd.read_csv('test_(1)_(2)_(1).csv')

In [ ]:
# 1. Preprocess the test data (similar to training data)
test_sentences = testData.Review.values
test_tokenized = [tokenizer.tokenize(sent)[:max_length] for sent in test_sentences]
test_tokenized = [['[CLS]'] + sent + ['[SEP]'] for sent in test_tokenized]

# 2. Convert to IDs and Pad
test_input_ids = [tokenizer.convert_tokens_to_ids(sent) for sent in test_tokenized]
test_input_ids = tf.keras.preprocessing.sequence.pad_sequences(test_input_ids, maxlen=max_length+2, truncating='post', padding='post')

# 3. Create Attention Masks
test_masks = np.array([[float(i>0) for i in seq] for seq in test_input_ids])

# 4. Predict
predictions = model.predict({'input_ids': test_input_ids, 'attention_mask': test_masks})
print(predictions)

In [ ]:
# 1. Convert logits to class labels (0 or 1)
# We take the index of the highest score (argmax) for each prediction
predicted_labels = np.argmax(predictions.logits, axis=1)

# 2. Add predictions back to the test dataframe for easy viewing
testData['Predicted_Rating'] = predicted_labels

# 3. Display a sample of the results
print("Sample Predictions:")
display(testData[['Review', 'Predicted_Rating']].head(10))

In [ ]:
sol = testData[['ID', 'Predicted_Rating']]

In [ ]:
sol.to_csv("./soln.csv", index=False)